# Nemotron-3-Embed: Advanced Fine-Tuning

Welcome! This notebook demonstrates how to build a robust embedding fine-tuning pipeline:
1. **Domain Filtering**: Select domain-specific text from `nvidia/Retrieval-Synthetic-NVDocs-v1`.
2. **Synthetic Data Generation**: Use a Qwen LLM to read document chunks and generate relevant queries as anchors.
3. **LoRA Fine-Tuning**: Fine-tune `nvidia/Nemotron-3-Embed-1B-BF16` efficiently using Hugging Face PEFT.
4. **LLM-as-a-Judge**: Use the Qwen model to evaluate retrieval relevance after training.

### Step 0: Install Dependencies

In [ ]:
!uv pip install -q sentence-transformers datasets accelerate peft trl transformers bitsandbytes

### Step 1: Domain Filtering

We load a slice of the dataset and filter it down to documents related to your chosen domain.

In [ ]:
from datasets import load_dataset

# Load a decent size slice to filter from
print("Loading dataset...")
raw_ds = load_dataset("nvidia/Retrieval-Synthetic-NVDocs-v1", split="train[:2000]")

# You can change the domain here! Some options:
# Medical: ["medical", "health", "clinical", "patient", "disease", "doctor", "medicine"]
# Finance: ["finance", "financial", "market", "economy", "investment", "banking", "stock", "trading"]
# Legal:   ["legal", "law", "court", "lawyer", "attorney", "contract", "litigation", "statute"]
domain_keywords = [
    "medical",
    "health",
    "clinical",
    "patient",
    "disease",
    "doctor",
    "medicine",
]


def is_target_domain(example):
    # Check raw text for keywords
    text_lower = example["text"].lower()
    if any(kw in text_lower for kw in domain_keywords):
        return True
    return False


domain_ds = raw_ds.filter(is_target_domain)
print(f"Original rows: {len(raw_ds)}, Filtered domain rows: {len(domain_ds)}")

# We will select a small subset (e.g., 100 rows) to keep generation and training fast for this tutorial
sample_ds = domain_ds.select(range(min(100, len(domain_ds))))

### Step 2: Synthetic Anchor Generation with Qwen

We load a Qwen LLM in 4-bit to generate questions. For each document, we extract a chunk and ask Qwen to generate a query that the chunk answers. This forms our `(anchor, positive)` training pair.

In [ ]:
import torch
import warnings
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig,
)
import random

warnings.filterwarnings("ignore", category=FutureWarning)

# Load Qwen in 4-bit to save VRAM
qwen_bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

# Options for models (Since we use NF4 4-bit, we can fit much larger models in VRAM!):
# "Qwen/Qwen2.5-1.5B-Instruct" (Requires ~1.5GB VRAM)
# "Qwen/Qwen2.5-3B-Instruct"   (Requires ~2.5GB VRAM)
# "Qwen/Qwen2.5-7B-Instruct"   (Requires ~4.5GB VRAM)
llm_id = "Qwen/Qwen2.5-7B-Instruct"

print(f"Loading {llm_id}...")
tokenizer = AutoTokenizer.from_pretrained(llm_id, clean_up_tokenization_spaces=False)
# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm_model = AutoModelForCausalLM.from_pretrained(
    llm_id, quantization_config=qwen_bnb, device_map="auto"
)
pipe = pipeline(
    "text-generation", model=llm_model, tokenizer=tokenizer, max_new_tokens=50
)


def generate_synthetic_pair(example, chunk_length=1500):
    doc = example["text"]
    # Take a random chunk if the document is too long
    if len(doc) > chunk_length:
        start_idx = random.randint(0, len(doc) - chunk_length)
        chunk = doc[start_idx : start_idx + chunk_length]
    else:
        chunk = doc

    prompt = f"<|im_start|>user\nRead the following document chunk and generate a single, short, specific question that is directly answered by it.\n\nDocument: {chunk}\n\nQuestion:<|im_end|>\n<|im_start|>assistant\n"

    # Generate
    outputs = pipe(
        prompt,
        do_sample=False,
        truncation=True,
        return_full_text=False,
        max_length=None,
    )
    question = outputs[0]["generated_text"].strip()

    return {"anchor": question, "positive": chunk}


print("Generating synthetic pairs (this may take a moment)...")
pair_ds = sample_ds.map(generate_synthetic_pair, remove_columns=sample_ds.column_names)
print("Generation Complete! Here are 3 examples:")
for i in range(min(3, len(pair_ds))):
    print(f"\n--- Example {i + 1} ---")
    print("Q:", pair_ds[i]["anchor"])
    print("A:", pair_ds[i]["positive"][:200], "...")

# Split into train and test sets
split_ds = pair_ds.train_test_split(test_size=0.1, seed=42)
train_ds = split_ds["train"]
eval_ds = split_ds["test"]

### Step 3: Load Nemotron and Apply LoRA

Now we load our embedding model and apply LoRA, allowing us to train the 1B model efficiently without running out of memory.

In [ ]:
import logging

# The model config has an unrecognized key for huggingface transformers, we ignore the warning:
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

from sentence_transformers import SentenceTransformer
from peft import get_peft_model, LoraConfig, TaskType
import gc

# Free some memory from LLM pipeline if possible, though pipeline stays resident.
torch.cuda.empty_cache()

embed_id = "nvidia/Nemotron-3-Embed-1B-BF16"
print(f"Loading base embedding model {embed_id}...")
embed_model = SentenceTransformer(
    embed_id, model_kwargs={"torch_dtype": torch.bfloat16}
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,
)

peft_model = get_peft_model(embed_model[0].auto_model, lora_config)
embed_model[0].auto_model = peft_model
print("\nLoRA injected! Trainable parameters:")
peft_model.print_trainable_parameters()

### Step 4: Fine-Tuning

We train using `MultipleNegativesRankingLoss`, leveraging the synthetically generated questions as our anchors.

In [ ]:
from sentence_transformers.sentence_transformer.losses import (
    MultipleNegativesRankingLoss,
)
from sentence_transformers.sentence_transformer.training_args import (
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.sentence_transformer.trainer import (
    SentenceTransformerTrainer,
)

loss = MultipleNegativesRankingLoss(embed_model)

args = SentenceTransformerTrainingArguments(
    output_dir="./nemotron-embed-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

trainer = SentenceTransformerTrainer(
    model=embed_model, args=args, train_dataset=train_ds, loss=loss
)

print("Starting training...")
trainer.train()
print("Training complete!")

### Step 5: LLM-as-a-Judge Evaluation

To ensure our model actually pulls relevant documents, we will take our evaluation dataset, compute embeddings, retrieve the closest document, and ask our Qwen model to judge if the retrieved document genuinely answers the query!

In [ ]:
import numpy as np

# Encode eval queries and all positive documents in eval set
print("Encoding evaluation set...")
eval_queries = eval_ds["anchor"]
eval_docs = eval_ds["positive"]

query_embs = embed_model.encode(eval_queries)
doc_embs = embed_model.encode(eval_docs)

# Compute dot product similarities
similarities = np.dot(query_embs, doc_embs.T)

judgments = []
correct_retrievals = 0

print("\nRunning LLM Judge on top retrieved documents...")
for i, query in enumerate(eval_queries):
    # Find the top 1 retrieved document for this query
    top_doc_idx = np.argmax(similarities[i])
    retrieved_doc = eval_docs[top_doc_idx]

    # Strict retrieval metric (Recall@1)
    if top_doc_idx == i:
        correct_retrievals += 1

    # LLM Judge
    prompt = f"<|im_start|>user\nGiven the Query and the Document, does the Document contain the correct answer or highly relevant information for the Query? Answer strictly 'Yes' or 'No'.\n\nQuery: {query}\nDocument: {retrieved_doc[:1000]}\n\nAnswer:<|im_end|>\n<|im_start|>assistant\n"
    outputs = pipe(
        prompt,
        do_sample=False,
        max_new_tokens=5,
        return_full_text=False,
        max_length=None,
    )
    answer = outputs[0]["generated_text"].strip().lower()
    judgments.append(1 if "yes" in answer else 0)

# Print metrics
recall_at_1 = correct_retrievals / len(eval_queries)
llm_judge_accuracy = sum(judgments) / len(judgments) if judgments else 0

print(f"\n--- Evaluation Results (on {len(eval_queries)} queries) ---")
print(f"Strict Recall@1: {recall_at_1 * 100:.2f}%")
print(f"LLM-Judge Relevance Score: {llm_judge_accuracy * 100:.2f}%")